# Training the Deepfake Detectors on Colab

Runs the full training pipeline on a free/Pro GPU and produces the two checkpoint
files the backend loads:

* `checkpoints/image_detector.pt` — used for images **and** video
* `checkpoints/audio_detector.pt` — used for audio

Until those exist, the platform runs in demonstration mode: every score is flagged
as non-evidential in the UI and in the generated PDF. That is deliberate.

**Before you start:** `Runtime -> Change runtime type -> GPU`. Training on the CPU
runtime is impractical.


## 1. Confirm you have a GPU


In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime > Change runtime type > GPU, then rerun.'


## 2. Get the code


In [ ]:
REPO = 'https://github.com/Mukesh182005/desktop-tutorial.git'
BRANCH = 'claude/project-from-pdf-6d11yl'

!git clone --branch $BRANCH --depth 1 $REPO project 2>/dev/null || echo 'already cloned'
%cd /content/project
!ls


## 3. Install dependencies

Colab already ships PyTorch, so only the pieces it lacks are installed here.


In [ ]:
!pip install -q facenet-pytorch librosa soundfile opencv-python-headless
!pip install -q -r backend/requirements.txt

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 4. Get the data

Two routes. **Route A** downloads the openly available Kaggle datasets — no approval
process, good for a first working model. **Route B** is for the datasets you
requested through an academic-use agreement (FaceForensics++, Celeb-DF, DFDC,
ASVspoof); upload those to Drive and point the paths at them.

Route A needs a Kaggle API token: kaggle.com -> Account -> Create New API Token.


### Route A — Kaggle (open access)


In [ ]:
from google.colab import files
import os, pathlib

# Upload kaggle.json when prompted.
if not pathlib.Path('/root/.kaggle/kaggle.json').exists():
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!python scripts/fetch_datasets.py faces-140k


### Route B — datasets from Drive

Skip if you used Route A.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# REAL_DIR = '/content/drive/MyDrive/datasets/ff++/original'
# FAKE_DIR = '/content/drive/MyDrive/datasets/ff++/manipulated'


## 5. Preprocess

Frames are sampled from videos, faces are detected and cropped, and the split is made
**by source video** — never by frame. Splitting by frame lets the same face appear in
both training and test, which inflates reported accuracy dramatically. Set the two
directories to match whichever route you took above.


In [ ]:
REAL_DIR = 'data/raw/faces-140k/real'      # or your Route B path
FAKE_DIR = 'data/raw/faces-140k/fake'

!python ml/preprocessing/build_face_dataset.py \
    --real-dir $REAL_DIR --fake-dir $FAKE_DIR \
    --output data/processed/faces --fps 1.0 --max-frames 32

import json
print(json.dumps(json.load(open('data/processed/faces/dataset_summary.json')), indent=2))


## 6. Train the image detector

Fine-tunes an ImageNet-pretrained backbone with a binary head, early stopping on
validation loss. Expect roughly 2–4 hours on a T4 for a full dataset; start with a
small `--epochs` to confirm the pipeline runs before committing to a long job.


In [ ]:
!python ml/training/train_image.py \
    --data data/processed/faces \
    --backbone efficientnet_b0 \
    --epochs 15 --batch-size 32 --lr 1e-4 --device cuda


## 7. Evaluate on the held-out test split

These are the numbers for your report and viva — accuracy, precision, recall, F1,
AUC-ROC, EER, plus a confusion matrix and ROC/PR curves. Quote *these*, not the
training-set numbers.


In [ ]:
!python ml/evaluation/evaluate.py \
    --model image --data data/processed/faces \
    --checkpoint checkpoints/image_detector.pt \
    --output reports/evaluation

from IPython.display import Image, display
for name in ['confusion_matrix', 'roc', 'pr_curve', 'score_distribution']:
    display(Image(f'reports/evaluation/image_{name}.png'))


## 8. Audio detector

Same shape: preprocess into fixed windows, then train the LCNN. The split is by
speaker, so the test set measures generalisation to voices the model never heard.


In [ ]:
BONAFIDE_DIR = 'data/raw/for-audio/real'
SPOOF_DIR    = 'data/raw/for-audio/fake'

!python scripts/fetch_datasets.py for-audio
!python ml/preprocessing/build_audio_dataset.py \
    --bonafide-dir $BONAFIDE_DIR --spoof-dir $SPOOF_DIR \
    --output data/processed/audio

!python ml/training/train_audio.py --data data/processed/audio --epochs 20 --device cuda
!python ml/evaluation/evaluate.py --model audio --data data/processed/audio \
    --checkpoint checkpoints/audio_detector.pt --output reports/evaluation


## 9. Confirm the backend will accept the checkpoints

This is the handoff check. It loads each file the way the API does and prints what
the UI and the PDF will say.


In [ ]:
!python scripts/verify_checkpoints.py


## 10. Save the checkpoints

Colab discards everything when the runtime ends. Copy the checkpoints to Drive (or
download them), then place them in `checkpoints/` on the machine running the app.

Do **not** commit them to git: they are large binaries and `.gitignore` excludes
`*.pt` on purpose.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/deepfake-checkpoints
!cp checkpoints/*.pt /content/drive/MyDrive/deepfake-checkpoints/
!cp checkpoints/*.json /content/drive/MyDrive/deepfake-checkpoints/ 2>/dev/null || true
!cp -r reports/evaluation /content/drive/MyDrive/deepfake-checkpoints/ 2>/dev/null || true
!ls -lh /content/drive/MyDrive/deepfake-checkpoints/


## 11. Test on samples the model has never seen

In-dataset test accuracy always flatters a detector. Make a handful of fakes with a
free online face-swap tool or a TTS voice cloner, run them through the deployed app,
and report how it did. Examiners find that far more convincing — and it is the
honest measure of whether the thing works.

---

### After this notebook

1. Put both `.pt` files in `checkpoints/` next to the app.
2. Run `make verify` — it should report 2/2 detectors trained.
3. Restart the API (and the Celery worker, if running). Models load once at startup.
4. The red 'demonstration mode' banner disappears and reports become evidential.
